# GPAP — Cross-GPU Benchmark

Measure **how fast GPAP propagates assumption sets on *this* GPU vs a sequential CPU baseline.**

For each bundled CNF the notebook runs the full GPAP pipeline over all (capped) ± variable
**pairs** and compares wall-clock against a sequential CPU [PySAT] loop over the *same* sets:

> **speedup = CPU_wall / GPU_wall = GPU sets/s ÷ CPU sets/s**

Triplets and a random-`L` assumption-size sweep are reported as supporting context. Results are
saved per-GPU to `benchmarks/colab_results/<gpu>.json`, so running this notebook on different
Colab GPUs lets you rank them (last cell).

**Before running:** `Runtime ▸ Change runtime type ▸ GPU` (T4/L4/A100…). The batch size `K`
auto-sizes to whatever VRAM the GPU has — no manual tuning.

## 1. Setup — clone repo + install deps
Torch ships preinstalled on Colab with the right CUDA build, so we only add `python-sat` + `tqdm`.

In [ ]:
import os, subprocess, sys

REPO_URL = "https://github.com/yogevshalmon/gpap.git"
if os.path.basename(os.getcwd()) != "gpap":
    if not os.path.isdir("gpap"):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, "gpap"], check=True)
    os.chdir("gpap")
subprocess.run(["git", "pull", "--ff-only"], check=False)   # refresh if you re-run later
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "python-sat", "tqdm"], check=True)
if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())
print("repo:", os.getcwd())

## 2. Detect the GPU
Confirms which accelerator this run used (recorded in the result file).

In [ ]:
import torch
from scripts.colab_benchmark import run_all, save_results, compare_saved, gpu_info

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
info = gpu_info(device)
for k, v in info.items():
    print(f"{k:>16}: {v}")
if info.get("total_gb") is None:
    print("\n  No CUDA GPU detected — set Runtime > Change runtime type > GPU and re-run cell 1.")

## 3. Run the benchmark
`quick=True` finishes in well under a minute (≈7 instances, pairs + a small L-sweep, no triplets).
Set `quick=False` for the full run (all 14 instances, triplets, full L-sweep `[2,5,10,20]`; a few minutes).

In [ ]:
payload = run_all(quick=True)

## 4. Save this GPU's result
Writes `benchmarks/colab_results/<gpu-slug>.json` (one file per GPU; re-runs overwrite).

In [ ]:
path = save_results(payload)

# Optional: download a copy to your machine.
try:
    from google.colab import files
    files.download(path)
except Exception:
    pass

## 5. (Optional) Commit the result back to the repo
Persist this GPU's numbers in version control so they accumulate across GPUs. Needs a GitHub
**Personal Access Token** with `repo` write scope. **Never hard-code a token you intend to keep** —
paste it here for the session only, or use Colab Secrets.

Uncomment, fill in, and run:

In [ ]:
# from getpass import getpass
# GITHUB_TOKEN = getpass("GitHub PAT: ")
# GIT_NAME, GIT_EMAIL = "Your Name", "you@example.com"
#
# subprocess.run(["git", "config", "user.name", GIT_NAME], check=True)
# subprocess.run(["git", "config", "user.email", GIT_EMAIL], check=True)
# subprocess.run(["git", "add", "benchmarks/colab_results/"], check=True)
# subprocess.run(["git", "commit", "-m", f"colab: {payload['gpu']['name']} benchmark result"], check=True)
# remote = f"https://{GITHUB_TOKEN}@github.com/yogevshalmon/gpap.git"
# subprocess.run(["git", "push", remote, "HEAD:main"], check=True)
# print("pushed")

## 6. Compare GPUs
Ranks every result JSON currently in `benchmarks/colab_results/` by peak pairs throughput.
Pull the repo (cell 1 re-run) to fetch results committed from other GPUs first.

In [ ]:
_ = compare_saved()